# Image-to-Image генерация
## Stable Diffusion + Face Restoration
### Бесплатный T4 GPU, модели без цензуры

In [ ]:
# Установка
!pip install diffusers transformers accelerate safetensors pillow requests

# Face restoration (CodeFormer)
!pip install -q git+https://github.com/sczhou/CodeFormer.git
!pip install -q basicsr facexlib

!pip install -q realesrgan  # upscale

In [ ]:
import torch
from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image
import requests
from io import BytesIO
from IPython.display import display, clear_output
import os, sys

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Устройство: {device}")
if device == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name}, VRAM: {gpu.total_memory/1024**3:.1f} GB")

In [ ]:
# === ВЫБОР МОДЕЛИ ===
# 1 - DreamShaper (лучшие лица, универсальная) ★
# 2 - Realistic Vision (фотореалистичные лица)
# 3 - Protogen (тёмный стиль, без цензуры)
# 4 - Anything v5 (аниме/арт, идеальные лица)

MODEL = 1

models = {
    1: "Lykon/dreamshaper-8",
    2: "SG161222/Realistic_Vision_V6.0_B1_noVAE",
    3: "darkstorm2150/Protogen_x3.4_Official_Release",
    4: "Linaqruf/anything-v5.0",
}

model_id = models[MODEL]

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None,
    requires_safety_checker=False
).to(device)

print(f"Загружена: {model_id}")

In [ ]:
# === ЗАГРУЗКА ИЗОБРАЖЕНИЯ ===
from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]
init_image = Image.open(filename).convert("RGB")
W, H = init_image.size
if W > 768 or H > 768:
    init_image.thumbnail((768, 768))
display(init_image)
print(f"Размер: {init_image.size}")

In [ ]:
# === ГЕНЕРАЦИЯ ===
prompt = ""               # что добавить/изменить
negative_prompt = ""       # что убрать (mutated hands, bad face и т.д.)
strength = 0.6             # 0.3 = близко к оригиналу, 0.9 = сильно изменить
steps = 30                 # качество (20-50)
guidance = 7.0             # 7-12

result = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=init_image,
    strength=strength,
    num_inference_steps=steps,
    guidance_scale=guidance
).images[0]

display(result)
result.save("output.png")
print("Сохранено: output.png")

In [ ]:
# === FACE RESTORATION (CodeFormer) ===
# Улучшает лица, если они кривые

RESTORE_FACE = True  # False = пропустить

if RESTORE_FACE:
    sys.path.append("/usr/local/lib/python3.*/site-packages")
    from codeformer_codeformer import CodeFormer
    from basicsr.archs.rrdbnet_arch import RRDBNet
    from realesrgan import RealESRGANer

    codeformer = CodeFormer(
        model_path="CodeFormer/weights/CodeFormer/codeformer.pth",
        upscale=1,
        fidelity=0.7
    )

    img = result.copy()
    restored = codeformer.enhance(img, has_aligned=False, only_center=False)[0]
    display(restored)
    restored.save("output_restored.png")
    print("Сохранено: output_restored.png (с восстановленными лицами)")
else:
    print("Пропущено")

In [ ]:
# Скачать результат
files.download("output.png")
import os
if os.path.exists("output_restored.png"):
    files.download("output_restored.png")

### Советы для лиц:
- **Prompt**: добавь `detailed face, perfect eyes, sharp facial features, high detail face`
- **Negative**: `mutated hands, bad face, deformed, blurry, bad anatomy, disfigured, ugly, worst quality`
- **Модели**: DreamShaper-8 (№1) даёт лучшие лица. Realistic Vision (№2) — фотореалистичные
- **CodeFormer** в последней ячейке чинит кривые лица
- Colab даёт T4 ~15ч/мес бесплатно